# Basic pyTorch tutorial

This tutorial is strongly inspired by this [tutorial](https://www.learnopencv.com/pytorch-for-beginners-image-classification-using-pre-trained-models/). Its purpose is to allow you to take a first step into the world of Pytorch!


In [ ]:
%pip install torch torchvision

: 


**Pre-trained models** will be used for **image classification**

Pre-trained models are neural network models driven on large data sets (such as ImageNet, which contains more than 14 million images from different classes).

The purpose of pre-trained models is to predict the class (or labels) of an input image. This process is called model inference. It includes the following main steps:

- Reading the input image
- Performing transformations on the image, for example resize, center crop, normalization, etc.
- Forward Pass: Use the pre-trained weights to find out the output vector. Each element in this output vector describes the confidence with which the model predicts the input image to belong to a particular class.
- Based on the scores obtained, display the predictions.

## step 1 : Loading a pre-trained network using TorchVision

TorchVision is a pytorch package. It consists of popular data sets, model architectures and common image transformations for computer vision. The different models available can be found here : [torchvision.models](https://pytorch.org/docs/stable/torchvision/models.html)

**How to decide which model to choose for a particular task ?**

The pre-trained models can be compared on the basis of the following criteria :

- Top-1 Error: A top-1 error occurs if the class predicted by a model with the highest confidence is not the same as the real class.
- Top-5 Error: A top-5 error occurs when the real class is not among the first 5 classes predicted by a model (sorted in terms of confidence).
- Inference time on CPU: The inference time is the time taken for the inference step of the model.
- Inference time on GPU
- Model size : Here the size represents the physical space occupied by the .pth file of the preformed model provided by PyTorch.

A good model will have a low Top-1 error, a low Top-5 error, a low inference time on the CPU and GPU and a low model size.

please define the pre-trained model you want to use using one of the predefined models in torchvision that you will call `model`

**indicators :** [`torchvision.models`](https://pytorch.org/docs/stable/torchvision/models.html)

**Solution :**

First of all, the different models and architectures available will be displayed.

In [ ]:
import torchvision.models as models
dir(models)

We can see that there is an entry `AlexNet` and another `alexnet`. Names beginning with a capital letter are Python classes (AlexNet) while alexnet is a function that returns the model of the AlexNet class. It is also possible for these functions to have different parameter sets. For example, densenet121, densenet161, densenet161, densenet161, densenet169, densenet201, all are instances of the DenseNet class but with a different number of layers - 121, 161, 169 and 201, respectively.

Here we choose to use the resnext101_32x8d network which has a Top-1 error and a Top-5 error below.

In [ ]:
model = models.resnext101_32x8d(pretrained=True) # downloading of the network and its weights
# model = models.resnext101_32x8d() # downloading of the network

#from torch import load
#model.load_state_dict(load('model/resnext101.pth')) # add of weights already downloaded

print('Network architecture :', '\n\n', model)

<img src="https://www.codeproject.com/KB/AI/1248963/resnet-r-700.png" width="900"/>

This model is composed of several layers, and has the particularity of introducing residual connections. Unlike **convolutional neural networks** which have a **linear architecture** (a stack of layers whose output is only connected to the next layer (architecture **A**), we will see these networks in more detail in the next tutorial), **in a residual network, the output of the previous layers is connected to the output of new layers to transmit them both to the next layer** (architecture **B**) :

<img src="https://makina-corpus.com/sites/default/files/import/blog_posts/classic-resnet-simplified.png" width="300"/>

the ResNet network is composed of 6 basic building blocks:

- a **convolution** (see [Conv2d](https://pytorch.org/docs/stable/nn.html#conv2d))
     - a convolution is an operation that serves at evaluating the match of a local area of the input with given features
     - the image is convolved by characteristic kernels (also called features)
<img src="https://stanford.edu/~shervine/teaching/cs-230/illustrations/convolution-layer-a.png" width="600"/>
<img src="https://sds-platform-private.s3-us-east-2.amazonaws.com/uploads/35_blog_image_11.png" width="600"/>


- a **Normalization** (see [BatchNorm2d](https://pytorch.org/docs/stable/nn.html#batchnorm2d))
    - Normalization of convolution-generated characteristic maps

- an **Activation** - Linear Rectification Unit function (see [ReLu()](https://pytorch.org/docs/stable/nn.html#relu))
    - allows to keep only positive values in the characteristic cards generated by the convolution
    - ReLU(x)=max(0,x) <img src="https://pytorch.org/docs/stable/_images/ReLU.png" width="400"/>
    - other types of activation :

![ReLU](https://qph.fs.quoracdn.net/main-qimg-07bc0ec05532caf5ebe8b4c82d0f5ca3)    


- a **Pooling**
    - a Max Pooling (see [MaxPool2d](https://pytorch.org/docs/stable/nn.html#maxpool2d))
        - allows you to keep only the maximum values in an area
        - reduces the characteristic map
    - an Average Pooling : voir [AdaptiveAvgPool2d](https://pytorch.org/docs/stable/nn.html#adaptiveavgpool2d)
        - allows you to keep only the averages of the values in a zone
        - reduces the characteristic map

<img src="https://www.researchgate.net/profile/Jelo_Salomon/publication/324728060/figure/download/fig13/AS:618948064202766@1524580126373/Max-and-Average-Pooling-Operation-2.png" width="400"/>

- **fully connected**
    - a network where all neurons of the output layer are connected to all neurons from the input layer
    - Linear (see [Linear](https://pytorch.org/docs/stable/nn.html#linear))
        - Applies a linear transformation to input data

## step 2 : Specify image transformations

Once we have the model, the next step is to transform the input image so that it has the right shape and other characteristics such as mean and standard deviation. These values must be similar to those used in the formation of the model. This ensures that the network will produce meaningful responses.

define the transformations to be performed on your image so that it is the right shape for your model.

**indicators :** [torchvision.models](https://pytorch.org/docs/stable/torchvision/models.html) + [torchvision.transforms](https://pytorch.org/docs/stable/torchvision/transforms.html)

**Solution :**

In [ ]:
# Here we define a variable transformation
#     which is a combination of all image transformations
#     to be performed on the input image.
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize(256),      # Resize the image to 256×256 pixels.
    transforms.CenterCrop(224),  # Crop the image to 224×224 pixels around the center.
    transforms.ToTensor(),       # Convert the image to PyTorch Tensor data type.
    transforms.Normalize(        # Normalize the image by adjusting its average and
                                 #     its standard deviation at the specified values.
    mean=[0.485, 0.456, 0.406],                
    std=[0.229, 0.224, 0.225]                  
    )])

## step 3 : Load the input image and pre-process it

- choose an image on the internet and save it
- execute the transformations defined in the previous step
- prepare a sample for transmission over the network

**indicators :** [urllib.request](https://docs.python.org/3/library/urllib.request.html) + [PIL](https://pillow.readthedocs.io/en/stable/reference/Image.html) + [torch.unsqueeze](https://pytorch.org/docs/stable/torch.html#torch.unsqueeze)

**Solution :**

In [ ]:

url_image = "https://www.hdwallpapersfreedownload.com/uploads/large/animals/wolf-image.jpg" # big image
url_image = "https://imgc.allpostersimages.com/img/print/posters/frank-lukasseck-gray-wolf_a-L-8655579-14258387.jpg" # smaller image
url_image = "https://4.bp.blogspot.com/-eSgcftxjewo/Tnbr-Cwm9iI/AAAAAAAAATQ/VC7Yvss2pFQ/s1600/bichon-frise-dog10.jpg" # smaller image
#url_image = "https://designerdoginfo.files.wordpress.com/2013/01/spoodle-puppy-4.jpg"
   
import urllib.request
urllib.request.urlretrieve(url_image, 'bichon.jpg')
# saves the image in the images folder under the name of img.jpg

In [ ]:
from PIL import Image # the Pillow module (PIL) is supported by default by TorchVision
from IPython.display import display

img = Image.open('bichon.jpg')

# displays the original image
display(img)

In [ ]:
# allows you to see the image at the input of the model
import torchvision
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def imshow(img_list, title=None):
    images = torchvision.utils.make_grid(img_list)
    
    fig = plt.figure(figsize=(5*len(img_list),5))
    """Imshow for Tensor."""
    inp = images.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    plt.xticks([]) ; plt.yticks([])
    if title is not None: plt.title(title)
    plt.show()

In [ ]:
# image preprocessing
img_t = transform(img)


In [ ]:

# sample so that it can be transmitted over the network
import torch
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
# device = torch.device('cpu') # uncomment to force CPU usage
device


In [ ]:
batch_t = torch.unsqueeze(img_t, 0)

# displays the input image of the model
imshow(batch_t)

## step 4 : Inference of the model

- put your model in evaluation mode and execute the inference
- using the `labels` list, containing the names of the 1000 imagenet classes, display the prediction of the model

In [ ]:
with open('imagenet_classes.txt') as f:
    labels = [line.strip() for line in f.readlines()]

In [ ]:
labels

**Solution :**

In [ ]:
model.eval() # evaluation mode
out = model(batch_t) # model inference

# out contains the model prediction for each of the 1000 imagenet classes
print(out.shape)

In [ ]:
# to display the model prediction
_, index = torch.max(out, 1)
percentage = torch.nn.functional.softmax(out, dim=1)[0] * 100
print(labels[index[0]], percentage[index[0]].item(), '%')

In [ ]:
# to display the other classes to which the model think the image belonged
_, indices = torch.sort(out, descending=True)
print([(labels[idx], percentage[idx].item()) for idx in indices[0][:5]])

## optional

display the prediction of different models for each image contained in the `images` folder

**Solution :**

In [ ]:
# Here we define several torchvision models
import torchvision.models as models

resnet18      = models.resnet18(pretrained=True)
alexnet       = models.alexnet(pretrained=True)
vgg16         = models.vgg16(pretrained=True)
squeezenet    = models.squeezenet1_0(pretrained=True)
shufflenet    = models.shufflenet_v2_x1_0(pretrained=True)
mobilenet     = models.mobilenet_v2(pretrained=True)
resnext50     = models.resnext50_32x4d(pretrained=True)
wide_resnet50 = models.wide_resnet50_2(pretrained=True)
mnasnet       = models.mnasnet1_0(pretrained=True)
resnext101    = models.resnext101_32x8d(pretrained=True)

import os
for i in np.sort(os.listdir('images')) :
    
    if i[-4:]=='.jpg' :
        print('\n\n', i[:-4], '\n', '-'*50, sep='')
        img = Image.open("images/%s"%i)
        
        img_t = transform(img)
        batch_t = torch.unsqueeze(img_t, 0)
        
        imshow(batch_t)
        
        for model, name in zip([resnet18, alexnet, vgg16, squeezenet,
                                shufflenet, mobilenet, resnext50, wide_resnet50, mnasnet,
                               resnext101],
                               ['resnet18', 'alexnet', 'vgg16', 'squeezenet',
                                'shufflenet', 'mobilenet', 'resnext50', 'wide_resnet50', 'mnasnet',
                                'resnext101']) :

            print(name, end=' ')
            model.eval()

            out = model(batch_t)
            percentage = torch.nn.functional.softmax(out, dim=1)[0] * 100
            _, indices = torch.sort(out, descending=True)

            for idx in indices[0][:1] :
                print(' %.1f%% - %s ' %(percentage[idx].item(), labels[idx]))

# Find a label in a torchvision dataset

display cat images from the STL10 torchvision dataset

**Solution :**

In [ ]:
image_datasets = torchvision.datasets.STL10('images', download=True, transform=transform)
dataloaders = torch.utils.data.DataLoader([image_datasets[x] for x in range(50)],
                                          batch_size=1,
                                          shuffle=True, num_workers=4)


tag = 'cat'
img_list = []
for img_t, _ in dataloaders.dataset:
    batch_t = torch.unsqueeze(img_t, 0)
    
    out = model(batch_t) # model inference
    _, index = torch.max(out, 1)
    percentage = torch.nn.functional.softmax(out, dim=1)[0] * 100
    #print(labels[index[0]], percentage[index[0]].item(), '%')
    if labels[index[0]][-len(tag):] == tag :
        img_list.append(img_t)

try : imshow(img_list, tag)
except : pass

# Image classification

In this tutorial we will study in more detail how to build a simple network using pytorch : [LeNet](http://yann.lecun.com/exdb/publis/pdf/lecun-01a.pdf) by Yann LeCun.
The code in this tutorial is based on a [example](https://github.com/pytorch/examples/blob/master/mnist/main.py) written in `pytorch`.

This network is a convolutional network that has the goal of classifiying digits.

This [tutorial](https://www.superdatascience.com/blogs/the-ultimate-guide-to-convolutional-neural-networks-cnn) will allow you to go into more detail in the explanation of this network

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

torch.manual_seed(1)
use_cuda = torch.cuda.is_available()

## step 1 : definition of the train image and evaluation

It is important to start by defining the images we will be working on.
Here, we will work on the MNIST dataset, containing images of handwritten numbers.

![MNIST](https://knowm.org/wp-content/uploads/Screen-Shot-2015-08-14-at-2.44.57-PM.png)

The purpose of the network will be to determine what number is written on the image. To do this, he will have to learn to correctly classify the images with a training dataset. He will then be able to test his learning with a second *validation* dataset.

(These two dataset must be different.)

In [ ]:
# size of the training batch
train_batch_size = 64
# size of the test batch
test_batch_size = 1000

# data transformation
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

kwargs = {'num_workers': 1, 'pin_memory': True} if use_cuda else {}


# dataset import
dataset = datasets.MNIST('images', download=False, transform=transform)

# number of images for training (80% of total data)
nb_train = int(0.8*len(dataset))
# number of images for the test (the rest of the data)
nb_test  = len(dataset)-nb_train

# separation of data into two datasets
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [nb_train, nb_test])


# Data loading
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=train_batch_size, shuffle=True, **kwargs)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=test_batch_size,  shuffle=True, **kwargs)

In [ ]:
print("The training set contains {} images, in {} batches of {} images".format(len(train_loader.dataset),
                                                                                      len(train_loader),
                                                                                      train_batch_size))
print("The test set contains {} images, in {} batches of {} images".format(len(test_loader.dataset),
                                                                               len(test_loader),
                                                                               test_batch_size))

## step 2 : Network

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 20, 5, 1) # convolution 1
        self.conv2 = nn.Conv2d(20, 50, 5, 1) # convolution 2
        self.fc1 = nn.Linear(4*4*50, 500) # fully connected 1
        self.fc2 = nn.Linear(500, 10) # fully connected 2

    def forward(self, x):
        # Extraction of characteristics 1
        x = F.relu(self.conv1(x)) # activation
        x = F.max_pool2d(x, 2, 2) # pooling
        
        # Extraction of characteristics 2
        x = F.relu(self.conv2(x)) # activation
        x = F.max_pool2d(x, 2, 2) # pooling
        x = x.view(-1, 4*4*50) # flattening
        
        # Classification
        x = F.relu(self.fc1(x)) # activation
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

the LeNet network consists of two main steps:
- the extraction of the image characteristics
- the classification of this image according to its characteristics

<img src="https://miro.medium.com/max/3712/1*7K4ZTTfZb-hbjoADbisHAg.png" width="600"/>

#### extraction of the image characteristics

to extract its characteristics the image will be modified in a series of steps: 

- a **convolution** (see [Conv2d](https://pytorch.org/docs/stable/nn.html#conv2d), [image parameters](https://github.com/vdumoulin/conv_arithmetic/blob/master/README.md))
     - a convolution is a function derived from two functions given by integration that expresses how the shape of one is modified by the other
     - the image is convoluted by characteristic detectors (called kernel)
     - Attention the entry each convolution layer must be the same size as the exit of the previous convolution layer
<img src="https://stanford.edu/~shervine/images/convolution-layer-a.png" width="600"/>
<img src="https://sds-platform-private.s3-us-east-2.amazonaws.com/uploads/35_blog_image_11.png" width="600"/>

- an **activation function** : Linear Rectification Unit function (see [ReLu()](https://pytorch.org/docs/stable/nn.html#relu))
    - to keep only the positive values in the characteristic cards generated by the convolution
    - ReLU(x)=max(0,x) <img src="https://pytorch.org/docs/stable/_images/ReLU.png" width="400"/>



- a **Pooling** : Max Pooling (see [MaxPool2d](https://pytorch.org/docs/stable/nn.html#maxpool2d))
    - to keep only the maximum values in an area
    - reduces the characteristic map
<img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/Max_pooling.png" width="300"/>

In [ ]:
model = Net()

In [ ]:
trial = 0
first_featuremap = 0
second_featuremap = 0

from Model_show import Plot_model
Plot_model(dataset, model.to('cpu'), trial, first_featuremap, second_featuremap)

### Classification

- **fully connected** - Linear (see [Linear](https://pytorch.org/docs/stable/nn.html#linear))
    - a network where all neurons of the output layer are connected to all neurons from the input layer
    - Applies a linear transformation to input data
<img src="https://jdlm.info/assets/driverless/27-fully-connected.png" width="900"/>     

- log_softmax (see [log_softmax](https://pytorch.org/docs/stable/nn.functional.html#log-softmax))

## step 3 : Network Training

- definition **Function loss** + find the right one!  [wiki](https://en.wikipedia.org/wiki/Convolutional_neural_network)

    The loss layer specifies how the network drive penalizes the difference between the expected and actual signal. It is normally the last layer in the network.

    Various loss functions adapted to different tasks can be used there [loss-functions](https://pytorch.org/docs/stable/nn.functional.html#loss-functions) :
    - The "Softmax" loss is used to predict only one of K mutually exclusive classes.
    - The sigmoid cross entropy loss is used to predict K values of independent probability in [0.1].
    - The Euclidean loss is used to regress to real values in [-inf ,inf].

The negative log likelihood loss -> allows to maximize the error when the right label has a low probability and to decrease it when the right label has a high probability!

- Backpropagation
    - the computation error spreads to the first layer of the network
<img src="https://d3i71xaburhd42.cloudfront.net/db39fd79bb591b04d33207992f6ccde03cabd861/7-Figure1-1.png" width="400"/>

    - To perform backpropagation in pytorch, the following steps are necessary for each iteration of the loop:

        1. **optimize.zero_grad()** : resets the gradients of each parameter to zero.
        2. **loss.backward()** : calculates the gradiants for each variable by backpropagation according to the loss, and stores them in the Variable object
        3. **optimize.step()** : Modifies each model parameter (network weight) to minimize loss.

In [ ]:
def train(model, device, train_loader, optimizer, epoch, log_interval):
    model.train() # put in training mode
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad() # important ! resets the gradients to 0
        output = model(data) # calculates the prediction
        loss = F.nll_loss(output, target) # calculates the error: The negative log likelihood loss.
        loss.backward() # drifts the graph
        optimizer.step() # performs an optimization step
        
        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))

In [ ]:
# we tell the model if we want to work on a GPU or a CPU
device = torch.device("cuda" if use_cuda else "cpu")
model = Net().to(device)

# the optimizer allows to calculate the network weights
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)

train(model, device, train_loader, optimizer, 0, 749)

## step 3 : Network Test

In [ ]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item() # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True) # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset), 100. * correct/len(test_loader.dataset)))
    return(100. * correct/len(test_loader.dataset))

we're going to test the model for 15 trials.

In [ ]:
N_test = 15
dict_test = dict(iter(test_loader))
inps = list(dict_test.keys())
inp = inps[0][:N_test]
targ = dict_test[inps[0]][:N_test]

i = inp.view(inp.shape[0], 1, inp.shape[2], inp.shape[3])

import torchvision.utils as vutils
grid_images = vutils.make_grid(i, nrow=N_test)
images = np.transpose(grid_images, (1,2,0))

plt.figure(figsize=(5*N_test, 5))
plt.imshow(images)
plt.xticks([]) ; plt.yticks([])
plt.show()

out = model(i.to(device))
pred = out.argmax(dim=1, keepdim=True)
print('target :', targ,
      '\nprediction :', pred.to('cpu').view(-1),
      '\nnumber of good predictions :', pred.eq(targ.to(device).view_as(pred)).sum().item(),
      '\ntotal number predictions :', N_test)      

## step 4 : evolution of accuracy at the heart of epochs

we train the model on several epochs by testing it for each epoch in order to see its evolution

In [ ]:
# we tell the model if we want to work on a GPU or a CPU
device = torch.device("cuda" if use_cuda else "cpu")
model = Net().to(device)
# the optimizer allows to calculate the network weights
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)

In [ ]:
#########################################################
# to reset the weights in the model to zero!
#########################################################
def weight_reset(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        m.reset_parameters()

model.apply(weight_reset)
#########################################################

In [ ]:
epochs = 15
log_interval = 749

list_accuracy = []
for epoch in range(1, epochs + 1):
    train(model, device, train_loader, optimizer, epoch, log_interval)
    accuracy = test(model, device, test_loader)
    list_accuracy.append(accuracy)

# Save model
torch.save(model.state_dict(),"model/mnist_cnn.pt")

In [ ]:
plt.plot(np.arange(1,len(list_accuracy)+1), list_accuracy, 'k')
plt.xlabel('epoch')
plt.ylabel('Accuracy')
plt.axis([0.9,len(list_accuracy)+0.1,90,100]);

In [ ]:
trial = 0
first_featuremap = 0
second_featuremap = 0

Plot_model(dataset, model.to('cpu'), trial, first_featuremap, second_featuremap)

## Visualization of the Kernels of the two convolutions of the network

The kernels used by the model are stored in:
- `model.conv1.weight` for the first convolution
- `model.conv2.weight` for the second convolution

In [ ]:
from Model_show import transform_img

**All kernels of convolution 1 :**

In [ ]:
fig, ax = plt.subplots(4,5, figsize=(15,15))
for num_w in range(model.conv1.weight.shape[0]) :
    w = model.conv1.weight[num_w].to('cpu')
    
    a, b = num_w//5, num_w%5
    ax[a][b].imshow(transform_img(w))
    ax[a][b].set_title(num_w+1)
    ax[a][b].set_xticks([]) ; ax[a][b].set_yticks([])
plt.show()

**All kernels of convolution 2 for the characteristic cards of the first kernel of convolution 1 :**

In [ ]:
fig, ax = plt.subplots(10,5, figsize=(15,15))
for num_w in range(model.conv2.weight.shape[0]) :
    w = model.conv2.weight[num_w][0].view(1,5,5).to('cpu')
    
    a, b = num_w//5, num_w%5
    ax[a][b].imshow(transform_img(w))
    ax[a][b].set_title(num_w+1)
    ax[a][b].set_xticks([]) ; ax[a][b].set_yticks([])
plt.tight_layout(pad=0)
plt.show()

## some bookkeeping for the notebook

In [ ]:
%load_ext watermark
%watermark -i -h -m -v -p numpy,matplotlib,torch,torchvision  -r -g -b